In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

importance_df = importance_df.sort_values(
    "Importance",
    ascending=True
).tail(15)

plt.figure(figsize=(8,6))

plt.barh(
    importance_df["Feature"],
    importance_df["Importance"]
)

plt.xlabel("Importance")

plt.title(
    "Top 15 Features (Tuned XGBoost)"
)

plt.tight_layout()

plt.savefig(
    "xgboost_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
gvws_processed = pd.read_parquet(
    "../data/processed/gvws_processed.parquet"
)

In [ ]:
windfarm_mean_processed = pd.read_parquet(
    "../data/processed/windfarm_mean_processed.parquet"
)

In [ ]:
uk_mean_processed = pd.read_parquet(
    "../data/processed/uk_mean_processed.parquet"
)

In [ ]:
#function for train, test and validate
def temporal_split(df):
    df = df.copy()

    train = df[
        (df["timestamp"] >= "2018-01-01") &
        (df["timestamp"] < "2024-01-01")
    ]

    val = df[
        (df["timestamp"] >= "2024-01-01") &
        (df["timestamp"] < "2025-01-01")
    ]

    test = df[
        (df["timestamp"] >= "2025-01-01") & 
        (df["timestamp"] < "2026-01-01")
    ]

    return train, val, test

In [ ]:
datasets = {
    "UK Mean": uk_mean_processed,
    "Wind-Farm Mean": windfarm_mean_processed,
    "GVWS": gvws_processed
}

split_datasets = {}

for name, df in datasets.items():

    train, val, test = temporal_split(df)

    split_datasets[name] = {
        "train": train,
        "val": val,
        "test": test
    }

    print(f"\n{name}")
    print(f"Train: {train.shape}")
    print(f"Validation: {val.shape}")
    print(f"Test: {test.shape}")

In [ ]:
#saving split data
for name, data in split_datasets.items():

    prefix = name.lower().replace(" ", "_")

    data["train"].to_parquet(
        f"../data/ttv/{prefix}_train.parquet",
        index=False
    )

    data["val"].to_parquet(
        f"../data/ttv/{prefix}_val.parquet",
        index=False
    )

    data["test"].to_parquet(
        f"../data/ttv/{prefix}_test.parquet",
        index=False
    )

In [ ]:
TARGET = "WIND"

def prepare_xy(df):

    X = df.drop(
        columns=[
            "timestamp",
            "WIND"
        ]
    )

    y = df["WIND"]

    return X, y

In [ ]:
from lightgbm import LGBMRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
import matplotlib.pyplot as plt

<h1>Presentation Evaluation

In [ ]:
def evaluate_representation(
    model,
    train_df,
    val_df,
    test_df,
    dataset_name,
    target_col="WIND",
    output_dir="../outputs/results/representations_results/"
):

    os.makedirs(output_dir, exist_ok=True)

    print(f"Running {dataset_name}")

    # -------------------------
    # Features / Target
    # -------------------------

    X_train = train_df.drop(
        columns=["timestamp", target_col]
    )

    y_train = train_df[target_col]

    X_val = val_df.drop(
        columns=["timestamp", target_col]
    )

    y_val = val_df[target_col]

    X_test = test_df.drop(
        columns=["timestamp", target_col]
    )

    y_test = test_df[target_col]

    # -------------------------
    # Fit model
    # -------------------------

    model.fit(
        X_train,
        y_train
    )

    preds = model.predict(X_test)

    # -------------------------
    # Metrics
    # -------------------------

    mae = mean_absolute_error(
        y_test,
        preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            preds
        )
    )

    r2 = r2_score(
        y_test,
        preds
    )

    metrics_df = pd.DataFrame({

        "Dataset": [dataset_name],
        "MAE": [mae],
        "RMSE": [rmse],
        "R2": [r2]

    })

    metrics_df.to_csv(

        os.path.join(
            output_dir,
            f"{dataset_name}_metrics.csv"
        ),

        index=False
    )

    # -------------------------
    # Prediction CSV
    # -------------------------

    pred_df = pd.DataFrame({

        "timestamp":
            test_df["timestamp"],

        "actual":
            y_test,

        "predicted":
            preds

    })

    pred_df.to_csv(

        os.path.join(
            output_dir,
            f"{dataset_name}_predictions.csv"
        ),

        index=False
    )

    # -------------------------
    # Actual vs Predicted
    # -------------------------

    plt.figure(figsize=(15,6))

    plt.plot(
        pred_df["timestamp"],
        pred_df["actual"],
        label="Actual"
    )

    plt.plot(
        pred_df["timestamp"],
        pred_df["predicted"],
        label="Predicted"
    )

    plt.legend()

    plt.title(
        f"{dataset_name} - Actual vs Predicted"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            output_dir,
            f"{dataset_name}_actual_vs_predicted.png"
        )

    )

    plt.close()

    # -------------------------
    # Residuals
    # -------------------------

    residuals = (
        y_test - preds
    )

    # Histogram

    plt.figure(figsize=(8,6))

    plt.hist(
        residuals,
        bins=50
    )

    plt.title(
        f"{dataset_name} Residual Distribution"
    )

    plt.xlabel("Residual")

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            output_dir,
            f"{dataset_name}_residual_histogram.png"
        )

    )

    plt.close()

    # Scatter

    plt.figure(figsize=(8,6))

    plt.scatter(
        preds,
        residuals,
        alpha=0.25
    )

    plt.axhline(
        y=0,
        color="red"
    )

    plt.xlabel("Predicted")

    plt.ylabel("Residual")

    plt.title(
        f"{dataset_name} Residual vs Predicted"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            output_dir,
            f"{dataset_name}_residual_scatter.png"
        )

    )

    plt.close()

    # -------------------------
    # Feature Importance
    # -------------------------

    if hasattr(model, "feature_importances_"):

        feat_imp = pd.DataFrame({

            "feature":
                X_train.columns,

            "importance":
                model.feature_importances_

        })

        feat_imp = feat_imp.sort_values(
            "importance",
            ascending=False
        )

        feat_imp.to_csv(

            os.path.join(
                output_dir,
                f"{dataset_name}_feature_importance.csv"
            ),

            index=False
        )

        plt.figure(figsize=(10,8))

        top_n = 20

        feat_imp.head(top_n).sort_values(
            "importance"
        ).plot.barh(

            x="feature",
            y="importance",
            legend=False

        )

        plt.title(
            f"{dataset_name} Feature Importance"
        )

        plt.tight_layout()

        plt.savefig(

            os.path.join(
                output_dir,
                f"{dataset_name}_feature_importance.png"
            )

        )

        plt.close()

    return {

        "Dataset": dataset_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2

    }

In [ ]:
#building lightGBM model function
results = []

for name, data in split_datasets.items():

    model = LGBMRegressor(

        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        random_state=42,
        verbose=-1

    )

    result = evaluate_representation(

        model=model,
        train_df=data["train"],
        val_df=data["val"],
        test_df=data["test"],
        dataset_name=name

    )

    #prediction_df[name] = result["pred_df"]

    results.append(result)

In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "../outputs/results/representations_results/representation_comparison.csv",
    index=False
)

results_df

In [ ]:
#high generation analysis
def high_generation_analysis(
    pred_df,
    dataset_name,
    output_dir="../data/results/representations_results/",
    quantile=0.90
):
    
    # Top generation periods
    threshold = pred_df["actual"].quantile(quantile)

    high_gen = pred_df[
        pred_df["actual"] >= threshold
    ].copy()

    # Metrics
    mae = mean_absolute_error(
        high_gen["actual"],
        high_gen["predicted"]
    )

    rmse = np.sqrt(
        mean_squared_error(
            high_gen["actual"],
            high_gen["predicted"]
        )
    )

    metrics_df = pd.DataFrame({
        "Dataset": [dataset_name],
        "Quantile": [quantile],
        "MAE": [mae],
        "RMSE": [rmse],
        "Num Events": [len(high_gen)]
    })

    metrics_df.to_csv(
        f"{output_dir}/{dataset_name}_high_generation_metrics.csv",
        index=False
    )

    # Plot
    plt.figure(figsize=(14,6))

    plt.plot(
        high_gen["timestamp"],
        high_gen["actual"],
        label="Actual",
        linewidth=2
    )

    plt.plot(
        high_gen["timestamp"],
        high_gen["predicted"],
        label="Predicted",
        linewidth=2
    )

    plt.title(
        f"{dataset_name}: Top {(1-quantile)*100:.0f}% High Generation Events"
    )

    plt.ylabel("Wind Generation (MW)")
    plt.xlabel("Time")

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        f"{output_dir}/{dataset_name}_high_generation_events.png",
        dpi=300
    )

    plt.close()

    return metrics_df

In [ ]:
gvws_pred = pd.read_csv(
    "../data/results/representations_results/GVWS_predictions.csv"
)

In [ ]:
gvws_high_metrics = high_generation_analysis(
    pred_df=gvws_pred,
    dataset_name="GVWS"
)

In [ ]:
uk_pred = pd.read_csv(
    "../data/results/representations_results/UK Mean_predictions.csv"
)

uk_pred["timestamp"] = pd.to_datetime(
    uk_pred["timestamp"]
)

wf_pred = pd.read_csv(
    "../data/results/representations_results/Wind-Farm Mean_predictions.csv"
)

wf_pred["timestamp"] = pd.to_datetime(
    wf_pred["timestamp"]
)

In [ ]:
uk_metrics = high_generation_analysis(
    pred_df=uk_pred,
    dataset_name="UK Mean"
)

wf_metrics = high_generation_analysis(
    pred_df=wf_pred,
    dataset_name="Wind-Farm Mean"
)

gvws_metrics = high_generation_analysis(
    pred_df=gvws_pred,
    dataset_name="GVWS"
)

In [ ]:
high_gen_results = pd.concat([

    uk_metrics,

    wf_metrics,

    gvws_metrics

], ignore_index=True)

high_gen_results

In [ ]:
plt.figure(figsize=(8,5))

plt.bar(
    high_gen_results["Dataset"],
    high_gen_results["RMSE"]
)

plt.ylabel("RMSE (MW)")

plt.title(
    "Forecast Performance During Top 10% Generation Events"
)

plt.tight_layout()

plt.savefig(
    "../data/results/representations_results/high_generation_rmse_comparison.png",
    dpi=300
)

plt.show()

In [ ]:
gvws_pred["timestamp"] = pd.to_datetime(
    gvws_pred["timestamp"]
)

In [ ]:
peak_idx = gvws_pred["actual"].idxmax()

peak_time = gvws_pred.loc[
    peak_idx,
    "timestamp"
]

In [ ]:
window = gvws_pred[
    (gvws_pred["timestamp"] >= peak_time - pd.Timedelta(days=3))
    &
    (gvws_pred["timestamp"] <= peak_time + pd.Timedelta(days=3))
]

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(
    window["timestamp"],
    window["actual"],
    label="Actual",
    linewidth=3
)

plt.plot(
    window["timestamp"],
    window["predicted"],
    label="Predicted",
    linewidth=2
)

plt.title(
    "GVWS Forecast During Peak Wind Generation Event"
)

plt.ylabel("Generation (MW)")

plt.legend()

plt.tight_layout()

plt.savefig(
    "../data/results/representations_results/GVWS_peak_event.png",
    dpi=300
)

plt.show()

<h1> Train, Test, Val data

In [ ]:
#train, val, test data
TARGET = "WIND"

def temporal_split(df):

    df = df.copy()

    train = df[
        (df["timestamp"] >= "2018-01-01")
        &
        (df["timestamp"] < "2024-01-01")
    ]

    val = df[
        (df["timestamp"] >= "2024-01-01")
        &
        (df["timestamp"] < "2025-01-01")
    ]

    test = df[
        (df["timestamp"] >= "2025-01-01")
        &
        (df["timestamp"] < "2026-01-01")
    ]

    return train, val, test

In [ ]:
gvws_processed["TARGET"] = gvws_processed["WIND"].shift(-24)
gvws_processed = gvws_processed.dropna()

def prepare_tree_data(
    train,
    val,
    test,
    target_col="TARGET"
):

    X_train = train.drop(
        columns=["timestamp", target_col]
    )

    y_train = train[target_col]

    X_val = val.drop(
        columns=["timestamp", target_col]
    )

    y_val = val[target_col]

    X_test = test.drop(
        columns=["timestamp", target_col]
    )

    y_test = test[target_col]

    return (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test
    )

<h1>day ahead persistance baseline

In [ ]:
train_df, val_df, test_df = temporal_split(
    gvws_processed
)

X_train, y_train, X_val, y_val, X_test, y_test = (
    prepare_tree_data(
        train_df,
        val_df,
        test_df,
        target_col="TARGET"
    )
)

In [ ]:
y_val_pred_persist = val_df["WIND"]
y_test_pred_persist = test_df["WIND"]

In [ ]:
persistence_results = {

    "Model": "Persistence",

    "MAE": mean_absolute_error(
        y_test,
        y_test_pred_persist
    ),

    "RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred_persist
        )
    ),

    "R2": r2_score(
        y_test,
        y_test_pred_persist
    )

}

In [ ]:
persistence_results

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,6))
plt.plot(y_test.values, label="Actual", alpha=0.8)
plt.plot(y_test_pred_persist.values, label="Predicted (Persistence)", alpha=0.8)
plt.title("Persistence: Actual vs Predicted Wind Generation")
plt.xlabel("Time")
plt.ylabel("Embedded Wind Generation")
plt.legend()
plt.show()

<h1> Traditional Physical Method (Power Curve)

In [ ]:
grid_capacity = pd.read_parquet(
    "../data/interim/weather_processed/grid_capacity.parquet"
)

In [ ]:
total_capacity = (
    grid_capacity[
        "Installed Capacity (MWelec)"
    ].sum()
)

In [ ]:
total_capacity

In [ ]:
def standard_power_curve(ws, total_capacity, v_in=3.0, v_rated=12.5, v_out=25.0):
    power = np.zeros_like(ws)
    
    # Partial load region (Cubic growth)
    mask_ramp = (ws >= v_in) & (ws < v_rated)
    power[mask_ramp] = total_capacity * ((ws[mask_ramp] - v_in) / (v_rated - v_in))**3
    
    # Rated load region
    mask_rated = (ws >= v_rated) & (ws < v_out)
    power[mask_rated] = total_capacity
    
    return power

In [ ]:
y_val_pred_physical = standard_power_curve(
    val_df["ws100"].values,
    total_capacity
)

y_test_pred_physical = standard_power_curve(
    test_df["ws100"].values,
    total_capacity
)

In [ ]:
plt.figure(figsize=(16,6))
plt.plot(y_test.values, label="Actual", alpha=0.8)
plt.plot(y_test_pred_physical, label="Predicted (Physical Curve)", alpha=0.8)
plt.title("Physical Model: Actual vs Predicted Wind Generation")
plt.xlabel("Time")
plt.ylabel("Embedded Wind Generation")
plt.legend()
plt.show()

In [ ]:
physical_results = {

    "Model": "Physical Curve",

    "MAE": mean_absolute_error(
        y_test,
        y_test_pred_physical
    ),

    "RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred_physical
        )
    ),

    "R2": r2_score(
        y_test,
        y_test_pred_physical
    )

}

In [ ]:
physical_results

<h1> Traditional Statistical Method (ARIMA)

In [ ]:
#!pip install statsmodels

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings("ignore")

In [ ]:
#fit base model purelu on training data
arima_model = ARIMA(y_train.values, order=(2, 0, 2))
arima_fit = arima_model.fit()

In [ ]:
y_test_pred_arima = []

In [ ]:
# Forecasting the validation set length
y_val_pred_arima = arima_fit.forecast(steps=len(y_val))

In [ ]:
# Forecasting the test set length
# Note: In a true walk-forward validation this would be updated continuously, 
# but for a static benchmark, we forecast the horizon.
y_test_pred_arima = arima_fit.forecast(steps=len(y_test))

In [ ]:
plt.figure(figsize=(16,6))
plt.plot(y_test.values, label="Actual", alpha=0.8)
plt.plot(y_test_pred_arima, label="Predicted (ARIMA)", alpha=0.8)
plt.title("ARIMA: Actual vs Predicted Wind Generation")
plt.xlabel("Time")
plt.ylabel("Embedded Wind Generation")
plt.legend()
plt.show()

In [ ]:
arima_results = {

    "Model": "ARIMA",

    "MAE": mean_absolute_error(
        y_test,
        y_test_pred_arima
    ),

    "RMSE": np.sqrt(
        mean_squared_error(
            y_test,
            y_test_pred_arima
        )
    ),

    "R2": r2_score(
        y_test,
        y_test_pred_arima
    )

}

In [ ]:
arima_results

In [ ]:
pd.DataFrame({

    "timestamp":
        test_df["timestamp"],

    "actual":
        y_test,

    "physical_pred":
        y_test_pred_physical,

    "phsical_pred":
        y_test_pred_physical,

    "arima_pred":
        y_test_pred_arima

}).to_csv(

    "../outputs/traditional_results.csv",

    index=False

)

<h1> XGBoost and LightGBM

In [ ]:
def evaluate_tree_model(
    model,
    model_name,
    train_df,
    val_df,
    test_df,
    output_dir="../outputs/models"
):

    os.makedirs(
        f"{output_dir}/{model_name}",
        exist_ok=True
    )

    (
        X_train,
        y_train,
        X_val,
        y_val,
        X_test,
        y_test
    ) = prepare_tree_data(
        train_df,
        val_df,
        test_df
    )

    model.fit(
        X_train,
        y_train
    )

    val_preds = model.predict(X_val)

    test_preds = model.predict(X_test)

    train_preds = model.predict(X_train)

    mae = mean_absolute_error(
        y_val,
        val_preds
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val,
            val_preds
        )
    )

    r2 = r2_score(
        y_val,
        val_preds
    )

    pred_df = pd.DataFrame({

        "timestamp":
            test_df["timestamp"],

        "actual":
            y_test,

        "predicted":
            test_preds

    })

    pred_df.to_csv(

        f"{output_dir}/{model_name}/predictions.csv",

        index=False
    )

    train_r2 = r2_score(
    y_train,
    train_preds
    )

    val_r2 = r2_score(
        y_val,
        val_preds
    )

    test_r2 = r2_score(
        y_test,
        test_preds
    )

    metrics = {

        "Model": model_name,

        "Train_R2": train_r2,
        "Val_R2": val_r2,
        "Test_R2": test_r2,

        "MAE": mae,
        "RMSE": rmse,
        "R2": test_r2

    }

    feature_model = None

    if hasattr(model, "feature_importances_"):

        feature_model = model

    elif hasattr(model, "best_estimator_") and hasattr(
        model.best_estimator_,
        "feature_importances_"
    ):

        feature_model = model.best_estimator_

        if feature_model is not None:

            importance_df = pd.DataFrame({

                "Feature":
                    X_train.columns,

                "Importance":
                    feature_model.feature_importances_

            })

            importance_df.sort_values(
                "Importance",
                ascending=False
            ).to_csv(

                f"{output_dir}/{model_name}/feature_importance.csv",

                index=False
            )

    return metrics

In [ ]:
# XGBoost Parameter Distribution
xgb_param_dist = {
    "n_estimators": [100, 300, 500, 1000],
    "max_depth": [4, 6, 8, 10], 
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# LightGBM Parameter Distribution
lgb_param_dist = {
    "n_estimators": [100, 300, 500, 1000],
    "max_depth": [-1, 4, 6, 8, 10], # -1 means no limit, letting num_leaves dictate growth
    "num_leaves": [31, 50, 70, 100], 
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor 

tscv = TimeSeriesSplit(n_splits=3)

tree_models = {

    "XGBoost":

        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=1000,
            max_depth=6,
            learning_rate=0.01,
            random_state=42,
            n_jobs=-1
        ),

    "XGB_Tuned":

        RandomizedSearchCV(
            estimator=XGBRegressor(
                    objective="reg:squarederror",
                    random_state=42
                ),
                param_distributions=xgb_param_dist,
                n_iter=20,                      
                scoring="neg_mean_squared_error",
                cv=tscv,
                verbose=1,
                n_jobs=-1,
                random_state=42
        ),

    "LightGBM":

        LGBMRegressor(
            n_estimators=1000,
            learning_rate=0.01,
            max_depth=6,
            random_state=42,
            n_jobs=-1
        ),

    "LightGBM_Tuned":

        RandomizedSearchCV(
            estimator=LGBMRegressor(
                objective="regression",
                random_state=42
            ),
            param_distributions=lgb_param_dist,
            n_iter=20,                      
            scoring="neg_mean_squared_error",
            cv=tscv,
            verbose=1,
            n_jobs=-1,
            random_state=42
        )

}

In [ ]:
train_df, val_df, test_df = temporal_split(
    gvws_processed
)

tree_results = []

for name, model in tree_models.items():

    result = evaluate_tree_model(

        model=model,

        model_name=name,

        train_df=train_df,

        val_df=val_df,

        test_df=test_df

    )

    tree_results.append(result)

tree_results = pd.DataFrame(
    tree_results
)

tree_results

<h1> LSTM and BiLSTM

In [ ]:
gvws_lstm = gvws_processed.copy()

gvws_lstm["TARGET"] = (
    gvws_lstm["WIND"].shift(-24)
)

gvws_lstm.dropna(inplace=True)

In [ ]:
train_df, val_df, test_df = temporal_split(
    gvws_lstm
)

In [ ]:
lag_cols = [

    "gen_lag_24",

    "WIND_roll_mean_24",
    "WIND_roll_std_24",

    "ws100_lag_24",
    "ws100_lag_168",

    "i10fg_lag_24",
    "i10fg_lag_168",

    "air_density_lag_24",
    "air_density_lag_168",

    "relative_humidity_lag_24",
    "relative_humidity_lag_168",

    "wind_shear_lag_24",
    "wind_shear_lag_168",

    "tp_lag_24",
    "tp_lag_168",

    "tcc_lag_24",
    "tcc_lag_168",

    "blh_lag_24",
    "blh_lag_168"

]

In [ ]:
X_train = train_df.drop(
    columns=[
        "timestamp",
        "TARGET"
    ] + lag_cols
)

X_val = val_df.drop(
    columns=[
        "timestamp",
        "TARGET"
    ] + lag_cols
)

X_test = test_df.drop(
    columns=[
        "timestamp",
        "TARGET"
    ] + lag_cols
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

feature_scaler = MinMaxScaler()

X_train_scaled = feature_scaler.fit_transform(
    X_train
)

X_val_scaled = feature_scaler.transform(
    X_val
)

X_test_scaled = feature_scaler.transform(
    X_test
)

In [ ]:
target_scaler = MinMaxScaler()

y_train_scaled = target_scaler.fit_transform(
    y_train.values.reshape(-1,1)
)

y_val_scaled = target_scaler.transform(
    y_val.values.reshape(-1,1)
)

y_test_scaled = target_scaler.transform(
    y_test.values.reshape(-1,1)
)

In [ ]:
lookback = 24

import numpy as np

def create_sequences(X, y, lookback):

    X_seq = []
    y_seq = []

    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])

    return (
        np.array(X_seq),
        np.array(y_seq)
    )

In [ ]:
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train_scaled,
    lookback
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val_scaled,
    lookback
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test_scaled,
    lookback
)

In [ ]:
print(X_train_seq.shape)
print(y_train_seq.shape)

In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
lstm_model = Sequential()

lstm_model.add(
    LSTM(
        32,
        input_shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        ),
        dropout=0.3,
        recurrent_dropout=0.3
    )
)

lstm_model.add(Dropout(0.3))

lstm_model.add(Dense(1))

In [ ]:
lstm_model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='mse'
)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tensorflow.keras.callbacks import Callback


In [ ]:
class VSCodeLivePlot(Callback):
    def on_train_begin(self, logs={}):
        self.losses = []
        self.val_losses = []
        # Create a persistent fig and axis to prevent flickering in VS Code
        self.fig, self.ax = plt.subplots(figsize=(9, 4.5))
        
    def on_epoch_end(self, epoch, logs={}):
        self.losses.append(logs.get('loss'))
        self.val_losses.append(logs.get('val_loss'))
        
        # Clear the old lines from the axis, not the whole window
        self.ax.clear()
        
        # Redraw the updated lines
        self.ax.plot(self.losses, label='Training Loss', color='#1f77b4', linewidth=2)
        self.ax.plot(self.val_losses, label='Validation Loss', color='#ff7f0e', linewidth=2)
        
        # Format the graph
        self.ax.set_title(f'LSTM Live Monitor - Epoch {epoch + 1}', fontsize=12, fontweight='bold')
        self.ax.set_xlabel('Epochs')
        self.ax.set_ylabel('Loss (MSE)')
        self.ax.grid(True, linestyle='--', alpha=0.5)
        self.ax.legend(loc='upper right')
        
        # Force VS Code to refresh the specific active cell output region
        clear_output(wait=True)
        display(self.fig)
        
    def on_train_end(self, logs={}):
        # Close the persistent figure when finished so it doesn't duplicate
        plt.close(self.fig)


In [ ]:
live_plot = VSCodeLivePlot()

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop, live_plot],
    verbose=0
)

In [ ]:
y_val_pred_scaled = lstm_model.predict(
    X_val_seq
)

In [ ]:
y_val_pred = target_scaler.inverse_transform(
    y_val_pred_scaled
)

y_val_actual = target_scaler.inverse_transform(
    y_val_seq
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(
    y_val_actual,
    y_val_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_val_actual,
        y_val_pred
    )
)

r2 = r2_score(
    y_val_actual,
    y_val_pred
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

<h1>BiLSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Bidirectional,
    LSTM,
    Dense,
    Dropout
)

bilstm_model = Sequential()

bilstm_model.add(
    Bidirectional(
        LSTM(32),
        input_shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    )
)

bilstm_model.add(Dropout(0.2))

bilstm_model.add(Dense(1))

bilstm_model.compile(
    optimizer=Adam(learning_rate=0.0003),
    loss='mse'
)

In [ ]:
history = bilstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop, live_plot],
    verbose=0
)

In [ ]:
y_val_pred_scaled = bilstm_model.predict(X_val_seq)

y_val_pred = target_scaler.inverse_transform(
    y_val_pred_scaled
)

y_val_actual = target_scaler.inverse_transform(
    y_val_seq
)

In [ ]:
mae = mean_absolute_error(
    y_val_actual,
    y_val_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_val_actual,
        y_val_pred
    )
)

r2 = r2_score(
    y_val_actual,
    y_val_pred
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

<h1> Perfect-future-weather benchmark

In [ ]:
benchmark_df = gvws_processed.copy()

In [ ]:
benchmark_df["TARGET"] = (
    benchmark_df["WIND"].shift(-24)
)

In [ ]:
weather_vars = [

    "ws100",
    "wd100",
    "wd10",

    "wind_shear",

    "relative_humidity",

    "air_density",

    "blh",

    "tcc",

    "tp",

    "ssrd",

    "i10fg"

]

In [ ]:
for col in weather_vars:

    benchmark_df[
        f"{col}_future_24h"
    ] = benchmark_df[col].shift(-24)

In [ ]:
benchmark_df[
    "wd100_sin_future_24h"
] = np.sin(
    np.deg2rad(
        benchmark_df[
            "wd100_future_24h"
        ]
    )
)

benchmark_df[
    "wd100_cos_future_24h"
] = np.cos(
    np.deg2rad(
        benchmark_df[
            "wd100_future_24h"
        ]
    )
)

In [ ]:
benchmark_df = benchmark_df.dropna()

In [ ]:
train_df, val_df, test_df = temporal_split(
    benchmark_df
)

In [ ]:
temporal_features = [

    "hour",
    "month",
    "dayofyear",

    "hour_sin",
    "hour_cos",

    "dayofyear_sin",
    "dayofyear_cos"

]

In [ ]:
current_weather_features = [

    "ws100",
    "wd100",
    "wd10",

    "wind_shear",

    "relative_humidity",

    "air_density",

    "blh",

    "tcc",

    "tp",

    "ssrd",

    "i10fg",

    "wd100_sin",
    "wd100_cos"

] + temporal_features

In [ ]:
X_train_c = train_df[
    current_weather_features
]

X_val_c = val_df[
    current_weather_features
]

X_test_c = test_df[
    current_weather_features
]

In [ ]:
y_train = train_df["TARGET"]

y_val = val_df["TARGET"]

y_test = test_df["TARGET"]

In [ ]:
future_weather_features = [

    "ws100_future_24h",

    "wd100_future_24h",

    "wd10_future_24h",

    "wind_shear_future_24h",

    "relative_humidity_future_24h",

    "air_density_future_24h",

    "blh_future_24h",

    "tcc_future_24h",

    "tp_future_24h",

    "ssrd_future_24h",

    "i10fg_future_24h",

    "wd100_sin_future_24h",

    "wd100_cos_future_24h"

] + temporal_features

In [ ]:
X_train_f = train_df[
    future_weather_features
]

X_val_f = val_df[
    future_weather_features
]

X_test_f = test_df[
    future_weather_features
]

In [ ]:
benchmark_model = LGBMRegressor(

    n_estimators=1000,

    learning_rate=0.01,

    max_depth=6,

    random_state=42,

    verbose=-1
)

In [ ]:
benchmark_model.fit(
    X_train_c,
    y_train
)

current_preds = benchmark_model.predict(
    X_test_c
)

In [ ]:
current_results = {

    "Scenario":
        "Current Weather",

    "MAE":
        mean_absolute_error(
            y_test,
            current_preds
        ),

    "RMSE":
        np.sqrt(
            mean_squared_error(
                y_test,
                current_preds
            )
        ),

    "R2":
        r2_score(
            y_test,
            current_preds
        )
}

In [ ]:
benchmark_model.fit(
    X_train_f,
    y_train
)

future_preds = benchmark_model.predict(
    X_test_f
)

In [ ]:
future_results = {

    "Scenario":
        "Perfect Future Weather",

    "MAE":
        mean_absolute_error(
            y_test,
            future_preds
        ),

    "RMSE":
        np.sqrt(
            mean_squared_error(
                y_test,
                future_preds
            )
        ),

    "R2":
        r2_score(
            y_test,
            future_preds
        )
}

In [ ]:
benchmark_results = pd.DataFrame([

    current_results,

    future_results

])

benchmark_results

In [ ]:
print(X_train_seq.shape)
print(X_val_seq.shape)
print(X_test_seq.shape)

In [ ]:
%reset